# Data File Analysis

This notebook demonstrates reading and analyzing CSV, JSON, Excel, and Word (.docx) files — common formats encountered in enterprise environments.

**Bingo tasks covered:**
- Inspect a CSV / JSON file and summarize key patterns
- Read / export an Excel file
- Read and export a .docx file

In [ ]:
import pandas as pd
import json
from pathlib import Path

DATA_DIR = Path("../data")

---
## 1. Inspect a CSV file and summarize key patterns

In [ ]:
df_onboarding = pd.read_csv(DATA_DIR / "hr_onboarding_log.csv")
df_onboarding.head()

In [ ]:
print(f"Total employees: {len(df_onboarding)}")
print(f"Completion rate: {df_onboarding['onboarding_completed'].mean():.0%}")
print(f"Avg days to complete: {df_onboarding['days_to_complete'].mean():.1f}")
print(f"Avg satisfaction score: {df_onboarding['satisfaction_score'].mean():.2f}")
print()
print("--- By Department ---")
print(df_onboarding.groupby("department").agg(
    count=("employee_id", "count"),
    avg_days=("days_to_complete", "mean"),
    avg_satisfaction=("satisfaction_score", "mean"),
    completion_rate=("onboarding_completed", "mean")
).round(2).to_string())

In [ ]:
# Key pattern: employees without mentors take longer and are less satisfied
print("--- Mentor Impact ---")
print(df_onboarding.groupby("mentor_assigned").agg(
    avg_days=("days_to_complete", "mean"),
    avg_satisfaction=("satisfaction_score", "mean"),
    completion_rate=("onboarding_completed", "mean")
).round(2).to_string())

---
## 2. Inspect a JSON file and summarize key patterns

In [ ]:
with open(DATA_DIR / "policy_queries.json") as f:
    queries = json.load(f)

df_queries = pd.DataFrame(queries)
df_queries.head()

In [ ]:
print(f"Total queries: {len(df_queries)}")
print(f"Helpful rate: {df_queries['helpful'].mean():.0%}")
print(f"Avg response time: {df_queries['response_time_ms'].mean():.0f} ms")
print()
print("--- Queries by Mode ---")
print(df_queries["mode"].value_counts().to_string())
print()
print("--- Most Queried Policies ---")
print(df_queries["policy_matched"].value_counts().to_string())
print()
print("--- Queries by Department ---")
print(df_queries["user_department"].value_counts().to_string())

---
## 3. Read an Excel file

In [ ]:
# Read both sheets from the Excel file
df_budget = pd.read_excel(DATA_DIR / "company_budget_and_training.xlsx", sheet_name="Department Budget")
df_training = pd.read_excel(DATA_DIR / "company_budget_and_training.xlsx", sheet_name="Training Completion")

print("=== Department Budget ===")
df_budget

In [ ]:
print(f"Total annual budget: ${df_budget['Annual Total'].sum():,.0f}")
print(f"Total headcount: {df_budget['Headcount'].sum()}")
print(f"Avg budget per employee: ${df_budget['Annual Total'].sum() / df_budget['Headcount'].sum():,.0f}")
print()
print("--- Budget per Employee by Department ---")
df_budget["Budget per Employee"] = (df_budget["Annual Total"] / df_budget["Headcount"]).round(0)
print(df_budget[["Department", "Annual Total", "Headcount", "Budget per Employee"]].to_string(index=False))

In [ ]:
print("=== Training Completion ===")
df_training

In [ ]:
print(f"Pass rate: {df_training['Passed'].mean():.0%}")
print(f"Avg score: {df_training['Score'].mean():.1f}")
print()
print("--- Pass Rate by Training Module ---")
print(df_training.groupby("Training Module").agg(
    avg_score=("Score", "mean"),
    pass_rate=("Passed", "mean")
).round(2).to_string())

---
## 4. Read a .docx file

In [ ]:
from docx import Document

doc = Document(DATA_DIR / "remote_work_policy_summary.docx")

# Display all paragraphs
for para in doc.paragraphs:
    if para.style.name.startswith("Heading"):
        print(f"\n{'#' * int(para.style.name[-1])} {para.text}")
    elif para.text.strip():
        print(para.text)

In [ ]:
# Extract tables from the docx
for i, table in enumerate(doc.tables):
    print(f"\n--- Table {i+1} ---")
    headers = [cell.text for cell in table.rows[0].cells]
    print(" | ".join(headers))
    print("-" * 60)
    for row in table.rows[1:]:
        print(" | ".join(cell.text for cell in row.cells))

In [ ]:
# Export: convert the .docx content to markdown
lines = []
for para in doc.paragraphs:
    if para.style.name.startswith("Heading"):
        level = int(para.style.name[-1]) if para.style.name[-1].isdigit() else 1
        lines.append(f"{'#' * level} {para.text}")
    elif para.style.name == "List Bullet":
        lines.append(f"- {para.text}")
    elif para.text.strip():
        lines.append(para.text)
    else:
        lines.append("")

md_output = "\n\n".join(lines)
output_path = DATA_DIR / "remote_work_policy_summary_exported.md"
output_path.write_text(md_output)
print(f"Exported to: {output_path}")
print(f"Length: {len(md_output)} characters")